## Info for dashboard

The notebook has the **same structure as 2. model_benchmarking**. In the **final section**, I **calculate the mean for continuous features and the mode for categorical features**. These statistics serve as essential **benchmarks**, allowing **future dashboard** users to **compare** their **input data against the typical profile of the training dataset**.


## Libraries and Settings

In [1]:
# =========================================
# 0. Imports
# =========================================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier

import shap
import warnings
# Silence specifically the LightGBM feature name validation warning
warnings.filterwarnings("ignore", message=".*X does not have valid feature names.*")

In [3]:

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## Data loading

In [4]:
# =========================================
# Load data
# =========================================
df = pd.read_csv("alzheimers_disease_data.csv")  

# Quick inspection
display(df.head())
print(df.shape)
print(df.info())

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


(2149, 35)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-n

##  Target and columns definition

In [5]:
# =========================================
# 2. Define columns manually
# =========================================
cat_cols = [
    "Gender", "Ethnicity", "EducationLevel", "Smoking",
    "FamilyHistoryAlzheimers", "CardiovascularDisease", "Diabetes", "Depression",
    "HeadInjury", "Hypertension", "MemoryComplaints", "BehavioralProblems",
    "Confusion", "Disorientation", "PersonalityChanges",
    "DifficultyCompletingTasks", "Forgetfulness"
]
cat_cols = [c for c in cat_cols if c in df.columns]

num_cols = [
    "Age", "BMI", "AlcoholConsumption", "PhysicalActivity", "DietQuality", "SleepQuality",
    "SystolicBP", "DiastolicBP", "CholesterolTotal", "CholesterolLDL",
    "CholesterolHDL", "CholesterolTriglycerides", "MMSE", "FunctionalAssessment", "ADL"
]
num_cols = [c for c in num_cols if c in df.columns]

In [6]:
# =========================================
#  Target definition
# =========================================

target_col = "Diagnosis"

# Se il target è stringa tipo 'Demented'/'NonDemented'
if df[target_col].dtype == "object":
    df[target_col] = df[target_col].astype("category").cat.codes

## Train/Test/ Validation split

In [7]:
# =========================================
# 4. Split features / target
# =========================================
id_cols = [col for col in ["PatientID", "patient_id", "DoctorID", "doctor_id"] if col in df.columns]
X = df.drop(columns=[target_col] + id_cols, errors="ignore")
y = df[target_col]


categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])


In [8]:
# =========================================
# 5. Model Pipeline & Data Splitting
# =========================================
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# Identify if a grouping column exists (to avoid data leakage between sessions of the same patient)
group_col = "PatientID" if "PatientID" in df.columns else None

if group_col is not None and df[group_col].nunique() < len(df):
    # Group-based split to ensure a patient's data is only in one set
    groups = df[group_col]

    # Step 1: Split off test set (20% of total data)
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    trainval_idx, test_idx = next(gss_test.split(X, y, groups=groups))

    X_trainval = X.iloc[trainval_idx].copy()
    y_trainval = y.iloc[trainval_idx].copy()
    groups_trainval = groups.iloc[trainval_idx].copy()

    X_test = X.iloc[test_idx].copy()
    y_test = y.iloc[test_idx].copy()

    # Step 2: Split trainval into train (75% of 80% = 60%) and val (25% of 80% = 20%)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx_rel, val_idx_rel = next(gss_val.split(X_trainval, y_trainval, groups=groups_trainval))

    X_train = X_trainval.iloc[train_idx_rel].copy()
    y_train = y_trainval.iloc[train_idx_rel].copy()

    X_val = X_trainval.iloc[val_idx_rel].copy()
    y_val = y_trainval.iloc[val_idx_rel].copy()

else:
    # Standard stratified split for independent samples (60/20/20 ratio)
    # First split: 80% train+val, 20% test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    # Second split: 75% of temp goes to train (60% total), 25% to validation (20% total)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
    )

# Display the final distribution of the datasets
print(f"{'='*30}")
print("DATA SPLIT SUMMARY")
print(f"{'='*30}")
print(f"Train set shape:      {X_train.shape} ({len(X_train)/len(df):.0%})")
print(f"Validation set shape: {X_val.shape} ({len(X_val)/len(df):.0%})")
print(f"Test set shape:       {X_test.shape} ({len(X_test)/len(df):.0%})")
print(f"{'='*30}")



DATA SPLIT SUMMARY
Train set shape:      (1289, 33) (60%)
Validation set shape: (430, 33) (20%)
Test set shape:       (430, 33) (20%)


In [9]:
# =========================================
#  Leakage Check: Patient Overlap 
# =========================================

id_col = "PatientID" 

if id_col in df.columns:
    # We use .index to look up the IDs in the original dataframe
    # This works whether you used GroupShuffleSplit or train_test_split
    train_ids = set(df.loc[X_train.index, id_col])
    test_ids = set(df.loc[X_test.index, id_col])
    val_ids = set(df.loc[X_val.index, id_col])
    
    # Calculate intersection
    overlap = train_ids.intersection(test_ids, val_ids)
    
    print("--- PATIENT OVERLAP ANALYSIS ---")
    print(f"Unique Patients in Training Set: {len(train_ids)}")
    print(f"Unique Patients in Test Set:     {len(test_ids)}")
    print(f"Unique Patients in Validation Set: {len(val_ids)}")
    
    if len(overlap) == 0:
        print(" SUCCESS: No patient overlap detected. Validation is clean.")
    else:
        print(f" CRITICAL WARNING: {len(overlap)} patients exist in BOTH sets!")
        print(f"Overlapping IDs: {list(overlap)[:10]}...") 
        
    # Check for multiple records per patient
    max_records = df[id_col].value_counts().max()
    if max_records > 1:
        print(f"Note: Some patients have up to {max_records} records in this dataset.")
else:
    print(f"Column '{id_col}' not found. Check your 'id_cols' list.")

--- PATIENT OVERLAP ANALYSIS ---
Unique Patients in Training Set: 1289
Unique Patients in Test Set:     430
Unique Patients in Validation Set: 430
 SUCCESS: No patient overlap detected. Validation is clean.


## Info for dashboard new cell

In [ ]:
# Mean for continous features and mode for categorical features
defaults_num = X_train[num_cols].mean().to_dict()
defaults_cat = X_train[cat_cols].mode().iloc[0].to_dict()

# Dictionary merging
all_defaults = {**defaults_num, **defaults_cat}


import pprint
pprint.pprint(all_defaults)

{'ADL': 4.954504193820691,
 'Age': 74.94957331264546,
 'AlcoholConsumption': 10.060665740281145,
 'BMI': 27.69596105705554,
 'BehavioralProblems': 0,
 'CardiovascularDisease': 0,
 'CholesterolHDL': 59.14403123954028,
 'CholesterolLDL': 124.14614012095403,
 'CholesterolTotal': 225.09747290218945,
 'CholesterolTriglycerides': 229.23819009037126,
 'Confusion': 0,
 'Depression': 0,
 'Diabetes': 0,
 'DiastolicBP': 90.26842513576416,
 'DietQuality': 4.996167433895887,
 'DifficultyCompletingTasks': 0,
 'Disorientation': 0,
 'EducationLevel': 1,
 'Ethnicity': 0,
 'FamilyHistoryAlzheimers': 0,
 'Forgetfulness': 0,
 'FunctionalAssessment': 5.079737730256702,
 'Gender': 1,
 'HeadInjury': 0,
 'Hypertension': 0,
 'MMSE': 14.885485070309867,
 'MemoryComplaints': 0,
 'PersonalityChanges': 0,
 'PhysicalActivity': 4.927858773107661,
 'SleepQuality': 7.057657327055819,
 'Smoking': 0,
 'SystolicBP': 133.83397982932505}
